<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study/13_Fruit_Images_for_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
출처:
https://www.kaggle.com/code/lojaaainmohaaamed/fruit-detection-using-yolov8s$0
```



In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

Mounted at /content/drive


In [ ]:
!kaggle datasets download mbkinaci/fruit-images-for-object-detection

Dataset URL: https://www.kaggle.com/datasets/mbkinaci/fruit-images-for-object-detection
License(s): CC0-1.0
100% 28.4M/28.4M [00:03<00:00, 9.82MB/s]



In [ ]:
!unzip -q fruit-images-for-object-detection.zip -d ./data

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import cv2
import random
import matplotlib.pyplot as plt
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import math
import os
for dirname, _, filenames in os.walk('/content/data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/content/data/test_zip/test/orange_89.jpg
/content/data/test_zip/test/apple_78.jpg
/content/data/test_zip/test/banana_84.xml
/content/data/test_zip/test/apple_88.xml
/content/data/test_zip/test/banana_85.xml
/content/data/test_zip/test/banana_87.jpg
/content/data/test_zip/test/banana_79.xml
/content/data/test_zip/test/apple_80.jpg
/content/data/test_zip/test/apple_87.jpg
/content/data/test_zip/test/apple_94.xml
/content/data/test_zip/test/orange_83.jpg
/content/data/test_zip/test/orange_84.xml
/content/data/test_zip/test/apple_85.xml
/content/data/test_zip/test/apple_78.xml
/content/data/test_zip/test/banana_92.jpg
/content/data/test_zip/test/apple_82.jpg
/content/data/test_zip/te

In [ ]:
os.listdir('./data')

['test_zip', 'train_zip']

In [ ]:
os.listdir('./data/train_zip')

['train']

In [ ]:
os.listdir('./data/train_zip/train')

['banana_65.xml',
 'apple_76.xml',
 'apple_11.xml',
 'apple_8.xml',
 'banana_24.jpg',
 'apple_61.xml',
 'orange_64.jpg',
 'apple_45.jpg',
 'orange_69.jpg',
 'apple_70.jpg',
 'banana_44.xml',
 'banana_56.jpg',
 'apple_55.jpg',
 'banana_41.xml',
 'orange_28.xml',
 'banana_76.jpg',
 'orange_52.xml',
 'apple_27.xml',
 'apple_59.xml',
 'apple_21.xml',
 'apple_32.xml',
 'banana_1.jpg',
 'banana_35.xml',
 'banana_4.jpg',
 'banana_54.xml',
 'banana_50.xml',
 'orange_57.xml',
 'mixed_8.jpg',
 'apple_69.xml',
 'orange_11.xml',
 'orange_8.jpg',
 'orange_58.jpg',
 'mixed_9.xml',
 'banana_47.jpg',
 'orange_33.xml',
 'orange_34.jpg',
 'apple_42.jpg',
 'orange_2.jpg',
 'orange_61.jpg',
 'orange_55.xml',
 'apple_23.jpg',
 'banana_48.xml',
 'mixed_2.xml',
 'apple_18.jpg',
 'apple_58.xml',
 'apple_45.xml',
 'orange_33.jpg',
 'banana_45.xml',
 'banana_69.xml',
 'banana_20.xml',
 'banana_29.xml',
 'banana_2.jpg',
 'mixed_15.xml',
 'orange_11.jpg',
 'banana_28.xml',
 'apple_73.jpg',
 'orange_19.xml',
 'ban

In [ ]:
if os.path.exists('/kaggle/working/raw_data'):
    shutil.rmtree('/kaggle/working/raw_data')

shutil.copytree(
    './data/train_zip/train',
    '/kaggle/working/raw_data'
)

'/kaggle/working/raw_data'

In [ ]:
os.makedirs('/kaggle/working/dataset/images/train', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels/train', exist_ok=True)

In [ ]:
classes = {
    'apple': 0,
    'banana': 1,
    'orange': 2,
    'mixed': 3
}

In [ ]:
raw_path = '/kaggle/working/raw_data'
img_out = '/kaggle/working/dataset/images/train'
lbl_out = '/kaggle/working/dataset/labels/train'

In [ ]:
os.makedirs(img_out, exist_ok=True)
os.makedirs(lbl_out, exist_ok=True)

def convert_bbox(size, box):
    dw = 1.0 / size[0] # 이미지 전체 가로 크기의 역수
    dh = 1.0 / size[1] # 이미지 전체 세로 크기의 역수

    # 1. 중심점(x, y) 구하기
    x = (box[0] + box[1]) / 2.0 # (xmin + xmax) / 2
    y = (box[2] + box[3]) / 2.0 # (ymin + ymax) / 2

    # 2. 박스의 실제 가로(w), 세로(h) 길이 구하기
    w = box[1] - box[0] # xmax - xmin
    h = box[3] - box[2] # ymax - ymin

    # 3. 0~1로 정규화해서 반환
    return x * dw, y * dh, w * dw, h * dh

skipped = 0
for file in os.listdir(raw_path):
    if not file.endswith('.xml'): # xml 파일만 골라내기.(xml 파일 나올 때까지 반복문을 돌리겠다.)
        continue

    tree = ET.parse(os.path.join(raw_path, file))
    root = tree.getroot()

    size = root.find('size')
    if size is None:
        skipped += 1
        continue

    w = int(size.find('width').text)
    h = int(size.find('height').text)

    if w == 0 or h == 0:
        skipped += 1
        continue

    img_name = root.find('filename').text
    img_path = os.path.join(raw_path, img_name)

    if not os.path.exists(img_path):
        skipped += 1
        continue

    shutil.copy(img_path, os.path.join(img_out, img_name))

    label_file = img_name.rsplit('.', 1)[0] + '.txt' # 예: apple1.jpg -> apple1.txt
    with open(os.path.join(lbl_out, label_file), 'w') as f: # xml 안의 모든 <object> 태그를 순회
        for obj in root.iter('object'):
            cls = obj.find('name').text
            if cls not in classes:
                continue

            # xml에서 픽셀 좌표 추출
            xmlbox = obj.find('bndbox')
            box = (
                float(xmlbox.find('xmin').text),
                float(xmlbox.find('xmax').text),
                float(xmlbox.find('ymin').text),
                float(xmlbox.find('ymax').text)
            )

            bb = convert_bbox((w, h), box)

            # txt 파일에 한줄 적기 -> [클래스ID 정수] [중심X] [중심Y] [W] [H]
            f.write(f"{classes[cls]} {' '.join(map(str, bb))}\n")
# 결론은..
# 모든 xml 파일을 읽어서 비정상 데이터는 거르고, 정상 데이터는 이미지 크기 비율에 맞춰 좌표를 변환한 뒤,
# YOLO 표준 포맷인 .txt 파일로 매핑해서 저장한다.

print(f"Skipped files due to errors: {skipped}")

Skipped files due to errors: 33


In [ ]:
len(os.listdir('/kaggle/working/dataset/images/train')),

(207,)

In [ ]:
len(os.listdir('/kaggle/working/dataset/labels/train'))

207

In [ ]:
train_imgs = os.listdir('/kaggle/working/dataset/images/train')

train_set, val_set = train_test_split(train_imgs, test_size=0.2, random_state=42)

os.makedirs('/kaggle/working/dataset/images/val', exist_ok=True)
os.makedirs('/kaggle/working/dataset/labels/val', exist_ok=True)

for img in val_set:
    shutil.move(
        f"/kaggle/working/dataset/images/train/{img}",
        f"/kaggle/working/dataset/images/val/{img}"
    )
    lbl = img.rsplit('.', 1)[0] + '.txt'
    shutil.move(
        f"/kaggle/working/dataset/labels/train/{lbl}",
        f"/kaggle/working/dataset/labels/val/{lbl}"
    )

In [ ]:
%%writefile data.yaml
path: /kaggle/working/dataset
train: images/train
val: images/val

names:
    0: apple
    1: banana
    2: orange
    3: mixed

Writing data.yaml


In [ ]:
# 뒤에 붙은 s의 의미: Small. 가볍지만 준수한 성능
model = YOLO('yolov8s.pt')

In [ ]:
# data: 설정 파일 경로
# 이 yaml 파일에는 딱 세 가지가 있습니다.
# 1. 학습 데이터 이미지 폴더 경로
# 2. 테스트 데이터 이미지 폴더 경로
# 3. 클래스 개수 및 클래스(['apple', 'banana', 'orange'])

# imgsz: 이미지 리사이즈 크기
# YOLOv8은 기본적으로 640 크기에서 최고의 성능을 내기 때문에 국룰입니다.
model.train(data='data.yaml', epochs=80, imgsz=640, batch=16)

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, pl

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7df1dc1caae0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [ ]:
# YOLO는 80번의 에폭을 돌면서 매번 성능을 기록하는데,
# 그중에서 가장 오차가 적고 정답률이 높았던 최고의 순간을 best.pt라는 가중치 파일로 저장해 둡니다.
best_model = YOLO('runs/detect/train/weights/best.pt')

# source: 검증용 과일 이미지들이 모여있는 폴더를 입력으로 줍니다.

# conf=0.25 (Confidence Threshold): 모델이 물체를 찾았을 때,
# "25% 이상의 확률로 확신하는 것만 꼬리표를 달아달라는 의미"
# 이 값을 너무 높이면 (0.9) 확실한 것만 맞히느라 박스를 몇개 못 그리고,
# 그렇다고 너무 낮추면 아무 데나 다 사과라고 박스를 쳐버립니다.
# 0.25는 국룰값입니다.

# save=True: 네모박스와 꼬리표를 예쁘게 그린 결과 이미지 파일을 폴더에 물리적으로 저장(True)합니다.
best_model.predict(source='/kaggle/working/dataset/images/val', conf=0.25, save=True)


image 1/42 /kaggle/working/dataset/images/val/apple_23.jpg: 640x640 3 apples, 11.0ms
image 2/42 /kaggle/working/dataset/images/val/apple_24.jpg: 640x448 3 apples, 1 banana, 76.2ms
image 3/42 /kaggle/working/dataset/images/val/apple_29.jpg: 640x640 1 apple, 15.1ms
image 4/42 /kaggle/working/dataset/images/val/apple_33.jpg: 640x640 1 apple, 11.2ms
image 5/42 /kaggle/working/dataset/images/val/apple_4.jpg: 448x640 2 apples, 54.6ms
image 6/42 /kaggle/working/dataset/images/val/apple_49.jpg: 640x608 1 apple, 46.1ms
image 7/42 /kaggle/working/dataset/images/val/apple_50.jpg: 512x640 3 apples, 46.7ms
image 8/42 /kaggle/working/dataset/images/val/apple_54.jpg: 608x640 1 apple, 46.2ms
image 9/42 /kaggle/working/dataset/images/val/apple_62.jpg: 448x640 1 apple, 12.4ms
image 10/42 /kaggle/working/dataset/images/val/apple_64.jpg: 640x640 5 apples, 11.9ms
image 11/42 /kaggle/working/dataset/images/val/apple_76.jpg: 512x640 1 apple, 9.3ms
image 12/42 /kaggle/working/dataset/images/val/apple_9.jpg: 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'apple', 1: 'banana', 2: 'orange', 3: 'mixed'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 2

In [ ]:
# 모델이 얼마나 정답을 잘 맞혔는지 통계학적으로 계산하는 단계.
# 아까 설정했던 data.yaml을 참조해서 정답지와 비교분석.
metrics = best_model.val(data='data.yaml', imgsz=640)

# 테스트 결과 보고서를 딕셔너리 형태로 출력
metrics.results_dict

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2659.7±995.1 MB/s, size: 133.8 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 42 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 42/42 22.0Mit/s 0.0s
val: /kaggle/working/dataset/images/val/apple_62.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6it/s 1.8s
                   all         42         89      0.881      0.936      0.965      0.725
                 apple         15         22      0.924      0.955      0.985      0.837
                banana         19         37      0.913      0.853      0.933      0.628
                orange         17         30      0.805          1      0.977       0.71
Speed: 7.7ms preprocess, 14.4ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/detect/val


{'metrics/precision(B)': 0.8807029890599111,
 'metrics/recall(B)': 0.9358658877073297,
 'metrics/mAP50(B)': 0.9651692313391523,
 'metrics/mAP50-95(B)': 0.7251913615691439,
 'fitness': 0.7251913615691439}